<a href="https://colab.research.google.com/github/baubyte/CienciaDeDatos/blob/main/Semana_4/Ejercicios/Taller_centros_de_salud_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller: Integración y análisis de datos de un conglomerado de centros de salud

## Contexto

Un conglomerado de centros de salud distribuidos en distintas provincias de Argentina necesita consolidar la información de sus consultas médicas para tomar decisiones de gestión. Los datos provienen de tres fuentes distintas:

- **consultas.csv**: registro de consultas médicas realizadas durante el año.
- **prestaciones.csv**: catálogo de prestaciones médicas con sus categorías y costos de referencia.
- **sucursales.txt**: listado de centros de salud con su ubicación geográfica.

El objetivo del taller es integrar estas fuentes, limpiar los datos y responder una pregunta de análisis asignada a cada grupo.

---

## Parte 1: Lectura y exploración de los datos

### 1.1 Importar las bibliotecas necesarias

In [1]:
# Importar la biblioteca necesaria
import pandas as pd
from google.colab import userdata, drive, files

#Montar Drive
drive.mount("/content/drive")



Mounted at /content/drive


In [14]:
url_dataset:str ="https://raw.githubusercontent.com/baubyte/CienciaDeDatos/refs/heads/main/Semana_4/Ejercicios/practica_centros.zip"
dir_dataset:str = 'drive/MyDrive/kaggle/datasets'
#Descargamos el dataset
!wget -nc {url_dataset} -P {dir_dataset}
# Descomprimir el dataset para acceder a los archivos CSV y TXT
!unzip -o {dir_dataset}"/practica_centros.zip" -d {dir_dataset}
#Elimnamos el zip
!rm {dir_dataset}"/practica_centros.zip"


File ‘drive/MyDrive/kaggle/datasets/practica_centros.zip’ already there; not retrieving.

Archive:  drive/MyDrive/kaggle/datasets/practica_centros.zip
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/consultas.csv  
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/prestaciones.csv  
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/sucursales.txt  
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/taller_centros_de_salud_v2.ipynb  


### 1.2 Leer el archivo de consultas

Leer el archivo `consultas.csv` en un DataFrame y explorar su contenido con `.shape`, `.dtypes`, `.head()` e `.info()`.

**Preguntas guía:**
- ¿Cuántas filas y columnas tiene el dataset?
- ¿Qué tipo de dato asignó Pandas a cada columna?
- ¿Hay valores faltantes? ¿En qué columnas?

In [ ]:
# Leer el archivo CSV de consultas
consultas = pd.read_csv(f"di/consultas.csv")

In [ ]:
# Explorar el DataFrame: shape, dtypes, head, info
consultas.shape


(650, 8)

In [ ]:
consultas.dtypes

,0
id_consulta,object
fecha,object
id_centro,object
id_prestacion,object
costo,object
obra_social,object
edad_paciente,int64
diagnostico,object


In [ ]:
consultas.head(20)

,id_consulta,fecha,id_centro,id_prestacion,costo,obra_social,edad_paciente,diagnostico
0,CON00001,22/01/2025,CS006,P005,12467,Swiss Medical,12,Contractura muscular
1,CON00002,28/12/2024,CS001,P002,8423,OSDE,54,Contractura muscular
2,CON00003,21/06/2024,CS003,P013,$8886,Galeno,36,Alergia estacional
3,CON00004,19/05/2024,CS003,P002,8076,Particular,59,Hipertensión arterial
4,CON00005,30/11/2024,CS004,P002,$9344,Particular,6,Diabetes tipo 2
5,CON00006,02/02/2025,CS003,P002,$ 10.181,PAMI,21,Gastritis
6,CON00007,06/09/2024,CS006,P005,$14317,Swiss Medical,69,Cefalea tensional
7,CON00008,04/07/2024,CS004,P005,$15483,Medifé,30,Hipertensión arterial
8,CON00009,2024-03-17,CS004,P005,$ 11.743,Particular,51,Faringitis
9,CON00010,24/01/2025,CS003,P003,13860,IOMA,75,Esguince


In [ ]:
consultas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_consulta    650 non-null    object
 1   fecha          650 non-null    object
 2   id_centro      650 non-null    object
 3   id_prestacion  650 non-null    object
 4   costo          650 non-null    object
 5   obra_social    650 non-null    object
 6   edad_paciente  650 non-null    int64 
 7   diagnostico    567 non-null    object
dtypes: int64(1), object(7)
memory usage: 40.8+ KB


In [ ]:
consultas.describe()

,edad_paciente
count,650.000000
mean,46.933846
std,26.362302
min,1.000000
25%,24.250000
50%,46.000000
75%,70.000000
max,92.000000


In [ ]:
consultas.isnull().sum()

,0
id_consulta,0
fecha,0
id_centro,0
id_prestacion,0
costo,0
obra_social,0
edad_paciente,0
diagnostico,83


### 1.3 Leer el archivo de prestaciones

Leer el archivo `prestaciones.csv` y explorar su contenido.

**Preguntas guía:**
- ¿Qué columnas contiene?
- ¿Qué columna podría usarse para vincularlo con el archivo de consultas?

In [ ]:
# Leer el archivo CSV de prestaciones
prestaciones = pd.read_csv('prestaciones.csv')


In [ ]:
prestaciones.head()

,id_prestacion,nombre_prestacion,categoria,costo_base
0,P001,Consulta clínica general,Clínica,8500
1,P002,Consulta pediátrica,Pediatría,9200
2,P003,Control cardiológico,Cardiología,15000
3,P004,Consulta dermatológica,Dermatología,11000
4,P005,Consulta traumatológica,Traumatología,13500


In [ ]:
# Explorar el DataFrame de prestaciones
prestaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_prestacion      15 non-null     object
 1   nombre_prestacion  15 non-null     object
 2   categoria          15 non-null     object
 3   costo_base         15 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 612.0+ bytes


### 1.4 Leer el archivo de centros de salud

Leer el archivo `sucursales.txt`. Tener en cuenta que está delimitado por tabulaciones.

**Preguntas guía:**
- ¿Qué separador se debe indicar para leer correctamente este archivo?
- ¿Los nombres de los centros están limpios o tienen algún problema?

In [ ]:
# Leer el archivo TXT de centros de salud
# Sugerencia: investigar el parámetro sep de pd.read_csv()


sucursales = pd.read_csv('sucursales.txt',sep='\t')


In [ ]:
sucursales.head()

,id_centro,nombre_centro,localidad,provincia,telefono
0,CS001,Centro de Salud Rivadavia,Ciudad Autónoma de Buenos Aires,Buenos Aires,011-4555-1234
1,CS002,Centro Médico del Sur,La Plata,Buenos Aires,0221-422-5678
2,CS003,Clínica San Martín,Córdoba,Córdoba,0351-489-0012
3,CS004,Centro de Salud Belgrano,Rosario,Santa Fe,0341-456-3344
4,CS005,Policlínico Norte,Mendoza,Mendoza,0261-411-7788


In [ ]:
sucursales.shape

(6, 5)

In [ ]:
sucursales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_centro      6 non-null      object
 1   nombre_centro  6 non-null      object
 2   localidad      6 non-null      object
 3   provincia      6 non-null      object
 4   telefono       6 non-null      object
dtypes: object(5)
memory usage: 372.0+ bytes


---

## Parte 2: Limpieza de datos

### 2.1 Limpiar la columna de costos

La columna `costo` del archivo de consultas no es numérica: contiene símbolos de pesos (`$`), espacios y puntos como separadores de miles.

**Tareas:**
1. Eliminar los caracteres no numéricos de la columna `costo`.
2. Convertir la columna a tipo numérico (`int` o `float`).
3. Verificar que la conversión fue exitosa.

In [ ]:
# Limpiar y convertir la columna de costos
# Sugerencia: investigar el método .str.replace() de Pandas
# Pista: hay que eliminar '$', '.' y espacios antes de convertir

consultas['costo'] = consultas['costo'].str.replace('$','').str.replace('.','').str.replace(' ','')
consultas['costo']  = pd.to_numeric(consultas['costo'])

consultas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_consulta    650 non-null    object
 1   fecha          650 non-null    object
 2   id_centro      650 non-null    object
 3   id_prestacion  650 non-null    object
 4   costo          650 non-null    int64 
 5   obra_social    650 non-null    object
 6   edad_paciente  650 non-null    int64 
 7   diagnostico    567 non-null    object
dtypes: int64(2), object(6)
memory usage: 40.8+ KB


### 2.2 Limpiar los nombres de los centros de salud

Algunos nombres de centros tienen espacios dobles o espacios al inicio/final.

**Tareas:**
1. Eliminar espacios extra en la columna `nombre_centro`.
2. Verificar el resultado comparando antes y después.

In [ ]:
# Limpiar los nombres de centros de salud
# Sugerencia: investigar .str.strip() y .str.replace() con expresiones regulares
sucursales['nombre_centro'] = sucursales['nombre_centro'].str.strip().str.replace(' +',' ')

### 2.3 Diagnóstico de los datos limpios

Redactar un breve resumen que describa:
- Qué problemas se encontraron en cada archivo.
- Cómo se resolvió cada uno.
- El estado final de cada DataFrame (cantidad de filas, columnas y tipos de datos).

# ¿Qué problemas se encontraron en cada archivo?




---

## Parte 3: Integración de los datos

### 3.1 Unir consultas con prestaciones

Realizar un join entre el DataFrame de consultas y el de prestaciones.

**Preguntas guía:**
- ¿Qué columna tienen en común ambos DataFrames?
- ¿Qué tipo de join conviene usar: `inner`, `left` o `right`? ¿Por qué?
- ¿Se pierden registros después del join? ¿Qué significaría si eso ocurriera?

In [ ]:
# Unir consultas con prestaciones
# Sugerencia: usar pd.merge() con los parámetros on= y how=


### 3.2 Unir con centros de salud

Agregar al DataFrame consolidado la información de los centros de salud (nombre del centro, localidad, provincia).

**Tareas:**
1. Realizar el segundo join.
2. Verificar la cantidad de filas resultante.
3. Explorar el DataFrame consolidado final.

In [ ]:
# Unir con centros de salud


In [ ]:
# Verificar el DataFrame consolidado



### 4.1 Pregunta de análisis

Cada grupo debe responder **una** de las siguientes preguntas:

1- ¿Cuál es la categoría de prestación más frecuente en cada centro de salud?

2- ¿Cuál es el costo promedio de consulta por obra social?

3- ¿Qué obra social concentra el mayor gasto total?

4-¿En qué provincia se realizaron más consultas?

---

### 4.2 Resolución

Utilizar `groupby()` y funciones de agregación (`.sum()`, `.mean()`, `.count()`, etc.) para responder la pregunta asignada.

In [ ]:
# Resolver la pregunta asignada al grupo


In [ ]:
# Tabla resumen con el resultado


### 4.3 Interpretación

Escribir una breve interpretación del resultado obtenido (3 a 5 oraciones). ¿Qué le diría este resultado al equipo de gestión de los centros de salud?

*Escribir la interpretación aquí.*